# Lab 01: Vectorized NumPy, Copy-on-Write Pandas & Analytical SQL Window Functions

**Companion to [Module 01](../curriculum/01-foundations/01_python_sql_and_data_stack.md)**: Run every cell to verify your data engineering foundations.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

# 1. Vectorized Z-score Standardization via Broadcasting
X = np.array([[10.0, 200.0], [14.0, 180.0], [8.0, 220.0], [16.0, 240.0]])
X_norm = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
assert np.allclose(X_norm.mean(axis=0), 0.0)
print('Vectorized Z-score shape:', X_norm.shape)

In [ ]:
# 2. Modern Pandas Copy-on-Write Safe Group Median Imputation
df = pd.DataFrame({
    'user_id': ['u1', 'u2', 'u3', 'u4'],
    'tier': ['pro', 'pro', 'free', 'free'],
    'spend': [120.0, np.nan, 20.0, np.nan],
})
df['spend_missing'] = df['spend'].isna().astype(int)
df['spend'] = df['spend'].fillna(df.groupby('tier')['spend'].transform('median'))
assert df['spend'].isna().sum() == 0
print(df)

In [ ]:
# 3. Analytical SQL Window Functions (LAG + Rolling Average + ROW_NUMBER) in SQLite
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE tx (user_id TEXT, ts INT, amount REAL)')
conn.executemany('INSERT INTO tx VALUES (?, ?, ?)', [
    ('u1', 1, 50.0), ('u1', 2, 70.0), ('u1', 3, 120.0), ('u2', 1, 30.0)
])
query = '''
SELECT
    user_id,
    ts,
    amount,
    LAG(amount, 1, 0.0) OVER (PARTITION BY user_id ORDER BY ts) AS prev_amount,
    AVG(amount) OVER (PARTITION BY user_id ORDER BY ts ROWS BETWEEN 1 PRECEDING AND CURRENT ROW) AS rolling_2_avg
FROM tx
ORDER BY user_id, ts;
'''
res_df = pd.read_sql_query(query, conn)
assert res_df.loc[2, 'prev_amount'] == 70.0
print(res_df)